# NYC Airbnb Pricing Analysis

**Business question:** what actually sets the price of a night in New York City?

I used the NYC Airbnb Open Data (Kaggle, 2019 snapshot) — 48,895 listings, one row per listing. Python for the analysis, SQL for the same questions in `../sql/airbnb_queries.sql`.

One honest note up front: this is a 2019 snapshot. There is no seasonality in it, and `reviews_per_month` is a proxy for demand, not actual bookings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the data

Download `AB_NYC_2019.csv` from Kaggle first (see `../data/DATA_SOURCE.md`) and save it in `data/`.

In [ ]:
df = pd.read_csv('../data/AB_NYC_2019.csv')
print(df.shape)
print(list(df.columns))

(48895, 16)
['id', 'name', 'host_id', 'host_name', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'availability_365', 'number_of_reviews', 'last_review', 'reviews_per_month']


## 2. Clean the price column

Price has a minimum of $0 in the raw file. A $0 listing is bad data, not a free room, so I drop rows with a missing or zero price. That removes 499 rows — 1.0% of the file — leaving 48,396 clean listings.

In [ ]:
before = len(df)
df = df[df['price'].notna() & (df['price'] > 0)].copy()
print('Rows before cleaning:', before)
print('Rows after cleaning:', len(df))
print('Rows dropped:', before - len(df))

Rows before cleaning: 48895
Rows after cleaning: 48396
Rows dropped: 499


## 3. Price distribution — the average lies

Mean price is about $152.70, but the median is $106, and the standard deviation (about $240) is bigger than the mean. A few very expensive listings pull the average up. For a 'typical' night, the median is the honest number.

In [ ]:
print('Mean price: $', round(df['price'].mean(), 2))
print('Median price: $', round(df['price'].median(), 2))
print('Std dev: $', round(df['price'].std(), 2))

Mean price: $ 152.7
Median price: $ 106.0
Std dev: $ 240.0


In [ ]:
df['price'].clip(upper=500).hist(bins=50)
plt.xlabel('Nightly price ($) — clipped at $500 for readability')
plt.ylabel('Listings')
plt.title('Nightly prices are heavily right-skewed')
plt.show()

## 4. Two boroughs run the market

Manhattan has 21,306 listings (44.0%) and Brooklyn 19,986 (41.3%). Together that is 85.3% of all listings. Queens has 11.7%, the Bronx 2.2%, Staten Island 0.8%.

In [ ]:
counts = df['neighbourhood_group'].value_counts()
print(counts[['Manhattan', 'Brooklyn']])
print((df['neighbourhood_group'].value_counts(normalize=True) * 100).round(1))

Manhattan    21306
Brooklyn     19986
Name: count, dtype: int64
Manhattan        44.0
Brooklyn         41.3
Queens           11.7
Bronx             2.2
Staten Island     0.8
Name: proportion, dtype: float64


## 5. Median price by borough — location sets the price

Median nightly price: Manhattan $149, Brooklyn $90, Queens $75, Staten Island $75, Bronx $65. A typical Manhattan listing costs about 2.3x a Bronx one.

In [ ]:
borough_median = df.groupby('neighbourhood_group')['price'].median().sort_values(ascending=False)
print(borough_median)

neighbourhood_group
Manhattan        149.0
Brooklyn          90.0
Queens            75.0
Staten Island    75.0
Bronx             65.0
Name: price, dtype: float64


In [ ]:
borough_median.plot(kind='bar')
plt.ylabel('Median nightly price ($)')
plt.title('Median price by borough')
plt.show()

## 6. Median price by room type

Median nightly price: entire home $160, private room $70, shared room $45. An entire home costs about 2.3x a private room and 3.6x a shared room.

In [ ]:
room_median = df.groupby('room_type')['price'].median().sort_values(ascending=False)
print(room_median)

room_type
Entire home/apt    160.0
Private room        70.0
Shared room           45.0
Name: price, dtype: float64


In [ ]:
room_median.plot(kind='bar')
plt.ylabel('Median nightly price ($)')
plt.title('Median price by room type')
plt.show()

## 7. A third of the market is multi-listing hosts

Hosts with more than one listing own 16,443 listings — 34.0% of the market. The biggest are Sonder (NYC) with 272 listings, Blueground with 180, and Kazuya with 103. This is not mostly spare rooms; a large share is professional operators.

In [ ]:
multi = df[df['calculated_host_listings_count'] > 1]
print('Listings owned by multi-listing hosts:', len(multi))
print('Share of market:', round(100 * len(multi) / len(df), 1), '%')
print(df.groupby('host_name')['id'].count().sort_values(ascending=False).head(3))

Listings owned by multi-listing hosts: 16443
Share of market: 34.0 %
host_name
Sonder (NYC)    272
Blueground      180
Kazuya          103
Name: id, dtype: int64


## 8. Reviews vs price — demand is a proxy, not bookings

`reviews_per_month` is a proxy for demand, not actual bookings. A listing with many reviews per month is in demand, but this data cannot say how many nights it was booked. Run the cell below to compare average reviews per month by room type.

In [ ]:
print(df.groupby('room_type')['reviews_per_month'].mean().round(2))
print('Correlation, reviews per month vs price:',
      round(df['reviews_per_month'].corr(df['price']), 3))

## 9. What I would tell a host

1. **Price by borough first, then by room type.** A private room in Manhattan can charge more than an entire home in the Bronx.
2. **Use the median, not the mean.** The mean ($152.70) is pulled up by a few luxury listings. The median ($106) is what a typical guest pays.
3. **Reviews are demand, not bookings.** A listing with many reviews per month is in demand — but this data cannot say how many nights it was actually booked.

## 10. Limitations

- 2019 snapshot: no seasonality, no trend over time, and prices today are different.
- `reviews_per_month` is a proxy for demand, not bookings or occupancy.
- Prices of $0 in the raw file are bad data, not free listings — they were removed.
- This is public listing data, not Airbnb's own records. It shows what hosts ask for, not what guests pay after fees.